In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")
error_logs = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes (Unix seconds -> UTC)
for df in (metric, trace, log, error_logs):
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)

# (1) Row counts for each file
row_counts = pd.DataFrame({
    "file": ["metric.csv", "trace.csv", "log.csv", "error_logs.csv"],
    "rows": [metric.shape[0], trace.shape[0], log.shape[0], error_logs.shape[0]]
})

# (2) Unique cmdb_id values found across all files (compact)
all_cmdb_ids = pd.Series(
    sorted(
        set(metric["cmdb_id"].dropna().unique())
        | set(trace["cmdb_id"].dropna().unique())
        | set(log["cmdb_id"].dropna().unique())
        | set(error_logs["cmdb_id"].dropna().unique())
    ),
    name="cmdb_id"
).to_frame()

# (3) metric: unique (cmdb_id, kpi_name) pairs with counts (top 50)
metric_pairs = (
    metric.groupby(["cmdb_id", "kpi_name"], observed=True)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(50)
)

# (4) trace: unique (cmdb_id, trace_name) pairs with counts (top 50)
trace_pairs = (
    trace.groupby(["cmdb_id", "trace_name"], observed=True)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(50)
)

# (5) log: unique (cmdb_id, log_name) pairs with counts (top 50)
log_pairs = (
    log.groupby(["cmdb_id", "log_name"], observed=True)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(50)
)

# Display compact summaries (as variables; IPython will show them)
row_counts, all_cmdb_ids, metric_pairs, trace_pairs, log_pairs

```
Out[1]:
```
summary = (
    "Summary of telemetry files:\n"
    "- Row counts: metric.csv=1900, trace.csv=3800, log.csv=460, error_logs.csv=0.\n"
    "- Unique cmdb_id values across files: [adservice, cartservice, checkoutservice, currencyservice, "
    "emailservice, frontend, frontend-external, frontendservice, paymentservice, productcatalogservice, "
    "recommendationservice, redis, root, shippingservice].\n"
    "- metric.csv (top 50 (cmdb_id, kpi_name) pairs): most pairs have count=25. Common KPIs observed per service "
    "include: cpu, diskio, error, latency-50, latency-90, mem.\n"
    "- trace.csv (top 50 pairs): most pairs have count=25. Typical trace names are of the form "
    "'trace.from_<service>.(duration_mean|duration_p95|error_rate|row_count)' and 'trace.from_root.*'. "
    "Checkoutservice-related traces appear prominently in the top rows.\n"
    "- log.csv (top 50 pairs): mostly 'log.error_count' and 'log.row_count' per service with count=25; "
    "redis is an exception with only 5 rows.\n"
    "- error_logs.csv contains no rows.\n"
    "Overall: telemetry is present for many services with consistent 25-row blocks per (service,metric/trace/log) pair; "
    "error_logs is empty and redis has reduced log coverage."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(             file  rows
0      metric.csv  1900
1       trace.csv  3800
2         log.csv   460
3  error_logs.csv     0,                   cmdb_id
0               adservice
1             cartservice
2         checkoutservice
3         currencyservice
4            emailservice
5                frontend
6       frontend-external
7         frontendservice
8          paymentservice
9   productcatalogservice
10  recommendationservice
11                  redis
12                   root
13        shippingservice,            cmdb_id    kpi_name  count
0        adservice         cpu     25
1        adservice      diskio     25
2        adservice       error     25
3        adservice  latency-50     25
4        adservice  latency-90     25
..             ...         ...    ...
45  paymentservice         cpu     25
46  paymentservice      diskio     25
47  paymentservice  latency-50     25
48  paymentservice  latency-90     25
49  paymentservice         mem     25

[50 rows x 3 columns],             cmdb_id                                trace_name  count
0   checkoutservice  trace.from_checkoutservice.duration_mean     25
1   checkoutservice   trace.from_checkoutservice.duration_p95     25
2   checkoutservice     trace.from_checkoutservice.error_rate     25
3   checkoutservice      trace.from_checkoutservice.row_count     25
4   checkoutservice  trace.from_frontendservice.duration_mean     25
..              ...                                       ...    ...
45     emailservice   trace.from_checkoutservice.duration_p95     25
46     emailservice     trace.from_checkoutservice.error_rate     25
47     emailservice      trace.from_checkoutservice.row_count     25
48     emailservice             trace.from_root.duration_mean     25
49     emailservice              trace.from_root.duration_p95     25

[50 rows x 3 columns],                   cmdb_id         log_name  count
0               adservice  log.error_count     25
1               adservice    log.row_count     25
2             cartservice  log.error_count     25
3             cartservice    log.row_count     25
4         checkoutservice  log.error_count     25
5         checkoutservice    log.row_count     25
6         currencyservice  log.error_count     25
7         currencyservice    log.row_count     25
8            emailservice  log.error_count     25
9            emailservice    log.row_count     25
10               frontend  log.error_count     25
11               frontend    log.row_count     25
12         paymentservice  log.error_count     25
13         paymentservice    log.row_count     25
14  recommendationservice  log.error_count     25
15  recommendationservice    log.row_count     25
19        shippingservice    log.row_count     25
18        shippingservice  log.error_count     25
17                  redis    log.row_count      5
16                  redis  log.error_count      5)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse existing 'metric' if present; otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")

# Ensure timestamp is parsed as UTC datetime (Unix seconds -> UTC)
if metric["timestamp"].dtype != "datetime64[ns, UTC]":
    metric["timestamp"] = pd.to_datetime(metric["timestamp"], unit="s", utc=True)

# Step 1: compute global P95 per (cmdb_id, kpi_name) using the entire series
p95 = (
    metric.groupby(["cmdb_id", "kpi_name"], observed=True)["value"]
    .quantile(0.95)
    .reset_index(name="p95_threshold")
)

# Step 2: filter metric to the incident window (UTC)
start = pd.to_datetime("2024-01-19 19:52:13", utc=True)
end = pd.to_datetime("2024-01-19 20:22:13", utc=True)
m_window = metric[(metric["timestamp"] >= start) & (metric["timestamp"] <= end)].copy()

# Step 3: join P95 thresholds and compute required aggregates per (cmdb_id, kpi_name) present in window
m_window = m_window.merge(p95, on=["cmdb_id", "kpi_name"], how="left")

grouped = (
    m_window.groupby(["cmdb_id", "kpi_name"], observed=True)
    .apply(
        lambda df: pd.Series(
            {
                "p95_threshold": df["p95_threshold"].iloc[0],
                "points_in_window": len(df),
                "anomaly_count": int((df["value"] >= df["p95_threshold"]).sum()),
                "earliest_anomaly_ts": (
                    df.loc[df["value"] >= df["p95_threshold"], "timestamp"].min()
                ),
                "max_value_in_window": float(df["value"].max()),
            }
        )
    )
    .reset_index()
)

# Format earliest_anomaly_ts as ISO strings in UTC (or None)
def fmt_ts(ts):
    if pd.isna(ts):
        return None
    return pd.to_datetime(ts).tz_convert("UTC").strftime("%Y-%m-%dT%H:%M:%SZ")

grouped["earliest_anomaly_ts"] = grouped["earliest_anomaly_ts"].apply(fmt_ts)

# Step 4: top 20 pairs sorted by anomaly_count desc
result_table = (
    grouped.sort_values(["anomaly_count", "points_in_window"], ascending=[False, False])
    .head(20)
    .loc[
        :,
        [
            "cmdb_id",
            "kpi_name",
            "p95_threshold",
            "points_in_window",
            "anomaly_count",
            "earliest_anomaly_ts",
            "max_value_in_window",
        ],
    ]
)

# Step 5: pairs where anomaly_count > 0 and anomaly_count == points_in_window (all in-window points anomalous), limit top 10
all_anomalous_pairs = (
    grouped[(grouped["anomaly_count"] > 0) & (grouped["anomaly_count"] == grouped["points_in_window"])]
    .sort_values(["anomaly_count", "points_in_window"], ascending=[False, False])
    .head(10)
    .loc[
        :,
        [
            "cmdb_id",
            "kpi_name",
            "p95_threshold",
            "points_in_window",
            "anomaly_count",
            "earliest_anomaly_ts",
            "max_value_in_window",
        ],
    ]
)

# Display compact results (IPython will render these variables)
result_table, all_anomalous_pairs

```
Out[2]:
```
summary = (
    "Summary (incident window 2024-01-19T19:52:13Z → 2024-01-19T20:22:13Z):\n"
    "- Global P95 thresholds were computed per (cmdb_id, kpi_name).\n"
    "- Many (cmdb_id, kpi_name) pairs had every in-window point >= P95 (anomaly_count = points_in_window = 25).\n"
    "  Representative fully-anomalous pairs include socket metrics for adservice, cartservice, checkoutservice, "
    "emailservice, paymentservice, productcatalogservice, recommendationservice, redis, shippingservice; and diskio/error "
    "metrics for several services. Earliest anomalies for most of these occurred at 2024-01-19T19:55:00Z.\n"
    "- Partial anomalies: currencyservice.socket had 12 anomalies out of 25 (earliest anomaly 2024-01-19T20:08:00Z).\n"
    "- Minor anomalies: adservice.cpu had 2 anomalies (earliest 2024-01-19T20:18:00Z, max value ≈ 1.006107), "
    "adservice.diskio had 2 anomalies (earliest 2024-01-19T20:00:00Z, max value ≈ 125415.647915).\n"
    "- Compact takeaway: the incident window shows widespread socket-related and diskio/error-related anomalies across "
    "many services (many with every sample anomalous), with most first anomalous samples at ~2024-01-19T19:55:00Z."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id kpi_name  p95_threshold  points_in_window  anomaly_count   earliest_anomaly_ts  max_value_in_window
2               adservice    error       0.000000                25             25  2024-01-19T19:55:00Z             0.000000
6               adservice   socket       4.000000                25             25  2024-01-19T19:55:00Z             4.000000
9             cartservice   diskio       0.000000                25             25  2024-01-19T19:55:00Z             0.000000
13            cartservice   socket       4.000000                25             25  2024-01-19T19:55:00Z             4.000000
19        checkoutservice   socket       9.000000                25             25  2024-01-19T19:55:00Z             9.000000
29           emailservice   diskio       0.000000                25             25  2024-01-19T19:55:00Z             0.000000
33           emailservice   socket       3.000000                25             25  2024-01-19T19:55:00Z             3.000000
36               frontend   diskio       0.000000                25             25  2024-01-19T19:55:00Z             0.000000
37               frontend    error       0.000000                25             25  2024-01-19T19:55:00Z             0.000000
46         paymentservice   diskio       0.000000                25             25  2024-01-19T19:55:00Z             0.000000
50         paymentservice   socket       3.000000                25             25  2024-01-19T19:55:00Z             3.000000
53  productcatalogservice   diskio       0.000000                25             25  2024-01-19T19:55:00Z             0.000000
57  productcatalogservice   socket       3.000000                25             25  2024-01-19T19:55:00Z             3.000000
60  recommendationservice   diskio       0.000000                25             25  2024-01-19T19:55:00Z             0.000000
64  recommendationservice   socket       4.000000                25             25  2024-01-19T19:55:00Z             4.000000
69                  redis   socket       4.000000                25             25  2024-01-19T19:55:00Z             4.000000
74        shippingservice   socket       2.000000                25             25  2024-01-19T19:55:00Z             2.000000
26        currencyservice   socket       7.000000                25             12  2024-01-19T20:08:00Z             7.000000
0               adservice      cpu       0.872827                25              2  2024-01-19T20:18:00Z             1.006107
1               adservice   diskio  119758.249063                25              2  2024-01-19T20:00:00Z        125415.647915,             cmdb_id kpi_name  p95_threshold  points_in_window  anomaly_count   earliest_anomaly_ts  max_value_in_window
2         adservice    error            0.0                25             25  2024-01-19T19:55:00Z                  0.0
6         adservice   socket            4.0                25             25  2024-01-19T19:55:00Z                  4.0
9       cartservice   diskio            0.0                25             25  2024-01-19T19:55:00Z                  0.0
13      cartservice   socket            4.0                25             25  2024-01-19T19:55:00Z                  4.0
19  checkoutservice   socket            9.0                25             25  2024-01-19T19:55:00Z                  9.0
29     emailservice   diskio            0.0                25             25  2024-01-19T19:55:00Z                  0.0
33     emailservice   socket            3.0                25             25  2024-01-19T19:55:00Z                  3.0
36         frontend   diskio            0.0                25             25  2024-01-19T19:55:00Z                  0.0
37         frontend    error            0.0                25             25  2024-01-19T19:55:00Z                  0.0
46   paymentservice   diskio            0.0                25             25  2024-01-19T19:55:00Z                  0.0)```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse existing 'trace' if present; otherwise load
try:
    trace
except NameError:
    trace = pd.read_csv("trace.csv")

# Ensure timestamp is parsed as UTC datetime (Unix seconds -> UTC)
if trace["timestamp"].dtype != "datetime64[ns, UTC]":
    trace["timestamp"] = pd.to_datetime(trace["timestamp"], unit="s", utc=True)

# Step 1: compute global P95 per (cmdb_id, trace_name) using the entire series (do NOT filter first)
p95_trace = (
    trace.groupby(["cmdb_id", "trace_name"], observed=True)["value"]
    .quantile(0.95)
    .reset_index(name="p95_threshold")
)

# Step 2: filter trace to the incident window (UTC)
start = pd.to_datetime("2024-01-19 19:52:13", utc=True)
end = pd.to_datetime("2024-01-19 20:22:13", utc=True)
t_window = trace[(trace["timestamp"] >= start) & (trace["timestamp"] <= end)].copy()

# Step 3: attach p95 thresholds and compute aggregates per (cmdb_id, trace_name) present in window
t_window = t_window.merge(p95_trace, on=["cmdb_id", "trace_name"], how="left")

grouped_trace = (
    t_window.groupby(["cmdb_id", "trace_name"], observed=True)
    .apply(
        lambda df: pd.Series(
            {
                "p95_threshold": df["p95_threshold"].iloc[0],
                "points_in_window": len(df),
                "anomaly_count": int((df["value"] >= df["p95_threshold"]).sum()),
                "earliest_anomaly_ts": df.loc[df["value"] >= df["p95_threshold"], "timestamp"].min(),
                "max_value_in_window": float(df["value"].max()),
            }
        )
    )
    .reset_index()
)

# Format earliest_anomaly_ts as ISO strings in UTC (or None)
def fmt_ts(ts):
    if pd.isna(ts):
        return None
    return pd.to_datetime(ts).tz_convert("UTC").strftime("%Y-%m-%dT%H:%M:%SZ")

grouped_trace["earliest_anomaly_ts"] = grouped_trace["earliest_anomaly_ts"].apply(fmt_ts)

# Step 4: top 20 pairs sorted by anomaly_count desc (tie-breaker by points_in_window desc)
result_table = (
    grouped_trace.sort_values(["anomaly_count", "points_in_window"], ascending=[False, False])
    .head(20)
    .loc[
        :,
        [
            "cmdb_id",
            "trace_name",
            "p95_threshold",
            "points_in_window",
            "anomaly_count",
            "earliest_anomaly_ts",
            "max_value_in_window",
        ],
    ]
)

# Step 5: short list (up to 10) of traces with name indicating error_rate or duration_p95/duration_mean and anomaly_count>0
mask_name = grouped_trace["trace_name"].str.contains("error_rate|duration_p95|duration_mean", case=False, na=False)
interesting_traces = (
    grouped_trace[mask_name & (grouped_trace["anomaly_count"] > 0)]
    .sort_values(["anomaly_count", "points_in_window"], ascending=[False, False])
    .head(10)
    .loc[:, ["cmdb_id", "trace_name", "earliest_anomaly_ts", "max_value_in_window"]]
)

# Display compact results
result_table, interesting_traces

```
Out[3]:
```
summary = (
    "Trace analysis summary (window 2024-01-19T19:52:13Z → 2024-01-19T20:22:13Z):\n"
    "- Global P95 was computed per (cmdb_id, trace_name).\n"
    "- Top anomalies: many trace.error_rate series (especially for checkoutservice and other services) "
    "had points_in_window=25 and anomaly_count=25. For these entries p95_threshold=0.0, earliest anomaly=2024-01-19T19:55:00Z, "
    "and max_value_in_window=0.0.\n"
    "- The separate list of interest (error_rate / duration_p95 / duration_mean with anomaly_count>0) is dominated by "
    "error_rate traces across checkoutservice, currencyservice, emailservice, frontendservice, etc., with earliest anomalies at 2024-01-19T19:55:00Z and max values 0.0.\n"
    "- Important caveat: many error_rate traces have a P95 of 0.0, so observed 0.0 values meet the '>= P95' rule and are counted as anomalies; this may be a threshold artifact rather than an indication of increased errors or durations.\n"
    "Takeaway: numerous trace.error_rate series are flagged as anomalous across multiple services starting ~2024-01-19T19:55:00Z, but the P95=0.0 condition suggests reviewing thresholds/semantics before concluding true error spikes."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id                                 trace_name  p95_threshold  points_in_window  anomaly_count   earliest_anomaly_ts  max_value_in_window
2   checkoutservice      trace.from_checkoutservice.error_rate            0.0                25             25  2024-01-19T19:55:00Z                  0.0
6   checkoutservice      trace.from_frontendservice.error_rate            0.0                25             25  2024-01-19T19:55:00Z                  0.0
10  checkoutservice                 trace.from_root.error_rate            0.0                25             25  2024-01-19T19:55:00Z                  0.0
14  checkoutservice        trace.to_checkoutservice.error_rate            0.0                25             25  2024-01-19T19:55:00Z                  0.0
18  checkoutservice        trace.to_currencyservice.error_rate            0.0                25             25  2024-01-19T19:55:00Z                  0.0
22  checkoutservice           trace.to_emailservice.error_rate            0.0                25             25  2024-01-19T19:55:00Z                  0.0
26  checkoutservice         trace.to_paymentservice.error_rate            0.0                25             25  2024-01-19T19:55:00Z                  0.0
30  checkoutservice  trace.to_productcatalogservice.error_rate            0.0                25             25  2024-01-19T19:55:00Z                  0.0
34  currencyservice      trace.from_checkoutservice.error_rate            0.0                25             25  2024-01-19T19:55:00Z                  0.0
38  currencyservice      trace.from_frontendservice.error_rate            0.0                25             25  2024-01-19T19:55:00Z                  0.0
42  currencyservice                 trace.from_root.error_rate            0.0                25             25  2024-01-19T19:55:00Z                  0.0
46     emailservice      trace.from_checkoutservice.error_rate            0.0                25             25  2024-01-19T19:55:00Z                  0.0
50     emailservice                 trace.from_root.error_rate            0.0                25             25  2024-01-19T19:55:00Z                  0.0
54  frontendservice      trace.from_frontendservice.error_rate            0.0                25             25  2024-01-19T19:55:00Z                  0.0
58  frontendservice                 trace.from_root.error_rate            0.0                25             25  2024-01-19T19:55:00Z                  0.0
62  frontendservice        trace.to_checkoutservice.error_rate            0.0                25             25  2024-01-19T19:55:00Z                  0.0
66  frontendservice        trace.to_currencyservice.error_rate            0.0                25             25  2024-01-19T19:55:00Z                  0.0
70  frontendservice        trace.to_frontendservice.error_rate            0.0                25             25  2024-01-19T19:55:00Z                  0.0
74  frontendservice  trace.to_productcatalogservice.error_rate            0.0                25             25  2024-01-19T19:55:00Z                  0.0
78  frontendservice  trace.to_recommendationservice.error_rate            0.0                25             25  2024-01-19T19:55:00Z                  0.0,             cmdb_id                                 trace_name   earliest_anomaly_ts  max_value_in_window
2   checkoutservice      trace.from_checkoutservice.error_rate  2024-01-19T19:55:00Z                  0.0
6   checkoutservice      trace.from_frontendservice.error_rate  2024-01-19T19:55:00Z                  0.0
10  checkoutservice                 trace.from_root.error_rate  2024-01-19T19:55:00Z                  0.0
14  checkoutservice        trace.to_checkoutservice.error_rate  2024-01-19T19:55:00Z                  0.0
18  checkoutservice        trace.to_currencyservice.error_rate  2024-01-19T19:55:00Z                  0.0
22  checkoutservice           trace.to_emailservice.error_rate  2024-01-19T19:55:00Z                  0.0
26  checkoutservice         trace.to_paymentservice.error_rate  2024-01-19T19:55:00Z                  0.0
30  checkoutservice  trace.to_productcatalogservice.error_rate  2024-01-19T19:55:00Z                  0.0
34  currencyservice      trace.from_checkoutservice.error_rate  2024-01-19T19:55:00Z                  0.0
38  currencyservice      trace.from_frontendservice.error_rate  2024-01-19T19:55:00Z                  0.0)```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse existing 'trace' if present; otherwise load
try:
    trace
except NameError:
    trace = pd.read_csv("trace.csv")

# Ensure timestamp is parsed as UTC datetime (Unix seconds -> UTC)
if trace["timestamp"].dtype != "datetime64[ns, UTC]":
    trace["timestamp"] = pd.to_datetime(trace["timestamp"], unit="s", utc=True)

# Focus on trace_names containing 'duration_mean' or 'duration_p95' (case-insensitive)
mask_duration = trace["trace_name"].str.contains("duration_mean|duration_p95", case=False, na=False)
trace_durations_full = trace[mask_duration].copy()

# Step 1: compute global P95 per (cmdb_id, trace_name) using the full durations series (do NOT filter by window)
p95_durations = (
    trace_durations_full.groupby(["cmdb_id", "trace_name"], observed=True)["value"]
    .quantile(0.95)
    .reset_index(name="p95_threshold")
)

# Step 2: filter trace to the incident window (UTC)
start = pd.to_datetime("2024-01-19 19:52:13", utc=True)
end = pd.to_datetime("2024-01-19 20:22:13", utc=True)
t_window = trace[(trace["timestamp"] >= start) & (trace["timestamp"] <= end)].copy()

# Keep only duration entries in the window
t_window_durations = t_window[t_window["trace_name"].str.contains("duration_mean|duration_p95", case=False, na=False)].copy()

# Step 3: attach p95 thresholds (computed on full series) and compute aggregates per (cmdb_id, trace_name) present in window
t_window_durations = t_window_durations.merge(p95_durations, on=["cmdb_id", "trace_name"], how="left")

def compute_group(df):
    p95 = df["p95_threshold"].iloc[0]
    pts = len(df)
    anom_cnt = int((df["value"] >= p95).sum()) if not pd.isna(p95) else 0
    earliest = df.loc[df["value"] >= p95, "timestamp"].min() if (not pd.isna(p95)) and (anom_cnt > 0) else pd.NaT
    max_val = float(df["value"].max())
    sev = max_val / (p95 if (pd.notna(p95) and p95 > 0) else 1e-9)
    return pd.Series({
        "p95_threshold": float(p95) if pd.notna(p95) else float("nan"),
        "points_in_window": pts,
        "anomaly_count": anom_cnt,
        "earliest_anomaly_ts": earliest,
        "max_value_in_window": max_val,
        "severity": float(sev),
    })

grouped = (
    t_window_durations.groupby(["cmdb_id", "trace_name"], observed=True)
    .apply(compute_group)
    .reset_index()
)

# Format earliest_anomaly_ts as ISO strings in UTC (or None)
def fmt_ts(ts):
    if pd.isna(ts) or ts is pd.NaT:
        return None
    return pd.to_datetime(ts).tz_convert("UTC").strftime("%Y-%m-%dT%H:%M:%SZ")

grouped["earliest_anomaly_ts"] = grouped["earliest_anomaly_ts"].apply(fmt_ts)

# Step 4A: Top 20 rows sorted by severity descending
result_top20 = (
    grouped.sort_values("severity", ascending=False)
    .head(20)
    .loc[:, ["cmdb_id", "trace_name", "p95_threshold", "points_in_window", "anomaly_count", "earliest_anomaly_ts", "max_value_in_window", "severity"]]
)

# Step 4B: from those results, short list (up to 10) of traces whose trace_name contains 'to_' and anomaly_count>0
downstream_mask = result_top20["trace_name"].str.contains(r"\bto_", case=False, na=False)
downstream_list = (
    result_top20[downstream_mask & (result_top20["anomaly_count"] > 0)]
    .loc[:, ["cmdb_id", "trace_name", "earliest_anomaly_ts", "severity"]]
    .head(10)
)

# Display compact results
result_top20, downstream_list

```
Out[4]:
```
Summary — duration anomalies in window 2024-01-19T19:52:13Z → 2024-01-19T20:22:13Z

Key findings:
- Highest-severity spikes (top results) are two calls to currencyservice initiated from root:
  - root · trace.to_currencyservice.duration_p95 — p95=0.000512, max=0.024876, severity ≈ 48.58, earliest anomaly 2024-01-19T19:59:00Z.
  - root · trace.to_currencyservice.duration_mean — p95=0.000139, max=0.004644, severity ≈ 33.44, earliest anomaly 2024-01-19T19:59:00Z.
- Other notable duration spikes (severity and earliest anomaly):
  - recommendationservice · trace.from_frontendservice.duration_p95 — severity ≈ 7.46, earliest 2024-01-19T20:15:00Z.
  - frontendservice · trace.to_recommendationservice.duration_p95 — severity ≈ 7.46, earliest 2024-01-19T20:15:00Z.
  - checkoutservice · trace.to_productcatalogservice.duration_mean / duration_p95 — severity ≈ 2.71 and 1.72, earliest 2024-01-19T20:02–20:04Z.
  - Several checkout/frontend/recommendation/productcatalog traces show moderate severity (≈1.4–1.6) with earliest anomalies between 20:01–20:17Z.
- Typical counts: points_in_window = 25 for these traces and anomaly_count = 2 for the top offenders (i.e., two in-window samples exceeded the global P95).

Downstream (trace_name contains "to_") traces with anomaly_count>0 (short list, up to 10):
- root · trace.to_currencyservice.duration_p95 — earliest 2024-01-19T19:59:00Z, severity ≈ 48.58
- root · trace.to_currencyservice.duration_mean — earliest 2024-01-19T19:59:00Z, severity ≈ 33.44
- frontendservice · trace.to_recommendationservice.duration_p95 — earliest 2024-01-19T20:15:00Z, severity ≈ 7.46
- checkoutservice · trace.to_productcatalogservice.duration_mean — earliest 2024-01-19T20:04:00Z, severity ≈ 2.71
- checkoutservice · trace.to_productcatalogservice.duration_p95 — earliest 2024-01-19T20:02:00Z, severity ≈ 1.72
- checkoutservice · trace.to_checkoutservice.duration_mean — earliest 2024-01-19T20:15:00Z, severity ≈ 1.60
- frontendservice · trace.to_recommendationservice.duration_mean — earliest 2024-01-19T20:13:00Z, severity ≈ 1.56
- recommendationservice · trace.to_recommendationservice.duration_mean — earliest 2024-01-19T20:17:00Z, severity ≈ 1.46
- frontendservice · trace.to_checkoutservice.duration_p95 — earliest 2024-01-19T20:10:00Z, severity ≈ 1.42
- checkoutservice · trace.to_currencyservice.duration_p95 — earliest 2024-01-19T20:01:00Z, severity ≈ 1.37

Takeaway:
- The most severe duration anomalies are calls to currencyservice (root→currencyservice) around 19:59Z and should be investigated first. Several downstream calls to recommendationservice and productcatalogservice also show smaller but noticeable spikes later in the window.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id                                      trace_name  p95_threshold  points_in_window  anomaly_count   earliest_anomaly_ts  max_value_in_window   severity
65                   root           trace.to_currencyservice.duration_p95       0.000512                25              2  2024-01-19T19:59:00Z             0.024876  48.578375
21        currencyservice                    trace.from_root.duration_p95       0.000512                25              2  2024-01-19T19:59:00Z             0.024876  48.578375
20        currencyservice                   trace.from_root.duration_mean       0.000139                25              2  2024-01-19T19:59:00Z             0.004644  33.438138
64                   root          trace.to_currencyservice.duration_mean       0.000139                25              2  2024-01-19T19:59:00Z             0.004644  33.438138
53  recommendationservice         trace.from_frontendservice.duration_p95       0.006304                25              2  2024-01-19T20:15:00Z             0.047050   7.463251
39        frontendservice     trace.to_recommendationservice.duration_p95       0.006304                25              2  2024-01-19T20:15:00Z             0.047050   7.463251
14        checkoutservice    trace.to_productcatalogservice.duration_mean       0.000022                25              2  2024-01-19T20:04:00Z             0.000060   2.714989
44  productcatalogservice        trace.from_checkoutservice.duration_mean       0.000022                25              2  2024-01-19T20:04:00Z             0.000060   2.714989
45  productcatalogservice         trace.from_checkoutservice.duration_p95       0.000032                25              2  2024-01-19T20:02:00Z             0.000055   1.719825
15        checkoutservice     trace.to_productcatalogservice.duration_p95       0.000032                25              2  2024-01-19T20:02:00Z             0.000055   1.719825
0         checkoutservice        trace.from_checkoutservice.duration_mean       0.008558                25              2  2024-01-19T20:15:00Z             0.013714   1.602457
6         checkoutservice          trace.to_checkoutservice.duration_mean       0.008558                25              2  2024-01-19T20:15:00Z             0.013714   1.602457
52  recommendationservice        trace.from_frontendservice.duration_mean       0.004521                25              2  2024-01-19T20:13:00Z             0.007069   1.563746
38        frontendservice    trace.to_recommendationservice.duration_mean       0.004521                25              2  2024-01-19T20:13:00Z             0.007069   1.563746
60  recommendationservice    trace.to_recommendationservice.duration_mean       0.003853                25              2  2024-01-19T20:17:00Z             0.005632   1.461598
54  recommendationservice  trace.from_recommendationservice.duration_mean       0.003853                25              2  2024-01-19T20:17:00Z             0.005632   1.461598
31        frontendservice           trace.to_checkoutservice.duration_p95       0.771660                25              2  2024-01-19T20:10:00Z             1.097132   1.421782
3         checkoutservice         trace.from_frontendservice.duration_p95       0.771660                25              2  2024-01-19T20:10:00Z             1.097132   1.421782
9         checkoutservice           trace.to_currencyservice.duration_p95       0.001153                25              2  2024-01-19T20:01:00Z             0.001578   1.368307
17        currencyservice         trace.from_checkoutservice.duration_p95       0.001153                25              2  2024-01-19T20:01:00Z             0.001578   1.368307,                   cmdb_id                                    trace_name   earliest_anomaly_ts   severity
65                   root         trace.to_currencyservice.duration_p95  2024-01-19T19:59:00Z  48.578375
64                   root        trace.to_currencyservice.duration_mean  2024-01-19T19:59:00Z  33.438138
39        frontendservice   trace.to_recommendationservice.duration_p95  2024-01-19T20:15:00Z   7.463251
14        checkoutservice  trace.to_productcatalogservice.duration_mean  2024-01-19T20:04:00Z   2.714989
15        checkoutservice   trace.to_productcatalogservice.duration_p95  2024-01-19T20:02:00Z   1.719825
6         checkoutservice        trace.to_checkoutservice.duration_mean  2024-01-19T20:15:00Z   1.602457
38        frontendservice  trace.to_recommendationservice.duration_mean  2024-01-19T20:13:00Z   1.563746
60  recommendationservice  trace.to_recommendationservice.duration_mean  2024-01-19T20:17:00Z   1.461598
31        frontendservice         trace.to_checkoutservice.duration_p95  2024-01-19T20:10:00Z   1.421782
9         checkoutservice         trace.to_currencyservice.duration_p95  2024-01-19T20:01:00Z   1.368307)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse existing metric and log if present; otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")
try:
    log
except NameError:
    log = pd.read_csv("log.csv")

# Ensure timestamps parsed as UTC datetimes (Unix seconds -> UTC)
for df in (metric, log):
    if df["timestamp"].dtype != "datetime64[ns, UTC]":
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)

# Incident window (UTC)
start = pd.to_datetime("2024-01-19 19:52:13", utc=True)
end = pd.to_datetime("2024-01-19 20:22:13", utc=True)

# -------------------------
# Part 1: metric.csv focused on currencyservice
# -------------------------

# Compute global P95 per kpi_name for currencyservice using the full metric series (rule: compute BEFORE filtering)
p95_currency = (
    metric[metric["cmdb_id"] == "currencyservice"]
    .groupby("kpi_name", observed=True)["value"]
    .quantile(0.95)
    .reset_index(name="p95_threshold")
)

# Filter metric to currencyservice rows in the incident window
m_window_cur = metric[
    (metric["cmdb_id"] == "currencyservice") & (metric["timestamp"] >= start) & (metric["timestamp"] <= end)
].sort_values("timestamp").copy()

# Attach p95 thresholds to windowed rows
m_window_cur = m_window_cur.merge(p95_currency, on="kpi_name", how="left")

# Helper to compute consecutive anomaly segments and earliest segment info for a sorted group
def metric_group_stats(df):
    # df assumed sorted by timestamp
    p95 = df["p95_threshold"].iloc[0]
    pts = len(df)
    # If p95 is NaN, treat as no anomalies
    if pd.isna(p95):
        return pd.Series({
            "p95_threshold": float("nan"),
            "points_in_window": pts,
            "anomaly_count": 0,
            "earliest_anomaly_ts": None,
            "max_value_in_window": float(df["value"].max()) if pts>0 else float("nan"),
            "number_of_consecutive_anomaly_segments": 0,
            "earliest_fault_start_ts": None,
            "length_of_earliest_fault_in_points": 0,
        })
    is_anom = df["value"] >= p95
    anom_count = int(is_anom.sum())
    earliest_ts = None
    if anom_count > 0:
        earliest_ts = pd.to_datetime(df.loc[is_anom, "timestamp"].min()).tz_convert("UTC").strftime("%Y-%m-%dT%H:%M:%SZ")
    max_val = float(df["value"].max()) if pts>0 else float("nan")
    # compute consecutive segments: a segment starts where is_anom is True and previous is False (or start)
    starts = is_anom & (~is_anom.shift(fill_value=False))
    number_segments = int(starts.sum())
    earliest_fault_start_ts = None
    length_earliest = 0
    if number_segments > 0:
        # find the first start index
        first_start_idx = starts.idxmax()  # label-based idx
        earliest_fault_start_ts = pd.to_datetime(df.loc[first_start_idx, "timestamp"]).tz_convert("UTC").strftime("%Y-%m-%dT%H:%M:%SZ")
        # To compute length, find contiguous True block length starting at that index
        # Use positional approach
        pos = df.index.get_loc(first_start_idx)
        # get boolean array
        arr = is_anom.to_numpy()
        # count from pos while True
        length = 0
        for v in arr[pos:]:
            if v:
                length += 1
            else:
                break
        length_earliest = int(length)
    return pd.Series({
        "p95_threshold": float(p95),
        "points_in_window": int(pts),
        "anomaly_count": anom_count,
        "earliest_anomaly_ts": earliest_ts,
        "max_value_in_window": max_val,
        "number_of_consecutive_anomaly_segments": number_segments,
        "earliest_fault_start_ts": earliest_fault_start_ts,
        "length_of_earliest_fault_in_points": length_earliest,
    })

# Group by kpi_name for currencyservice in-window and compute stats
if m_window_cur.shape[0] > 0:
    currency_metric_kpis = (
        m_window_cur.sort_values(["kpi_name", "timestamp"])
        .groupby("kpi_name", observed=True)
        .apply(metric_group_stats)
        .reset_index()
    )
else:
    # empty result structure
    currency_metric_kpis = pd.DataFrame(columns=[
        "kpi_name","p95_threshold","points_in_window","anomaly_count","earliest_anomaly_ts",
        "max_value_in_window","number_of_consecutive_anomaly_segments","earliest_fault_start_ts","length_of_earliest_fault_in_points"
    ])

# -------------------------
# Part 2: log.csv focused on currencyservice
# -------------------------

# Compute global P95 per (cmdb_id, log_name) using full log.csv (do NOT filter first)
p95_log = (
    log.groupby(["cmdb_id", "log_name"], observed=True)["value"]
    .quantile(0.95)
    .reset_index(name="p95_threshold")
)

# Filter log to currencyservice rows in the incident window
log_window_cur = log[
    (log["cmdb_id"] == "currencyservice") & (log["timestamp"] >= start) & (log["timestamp"] <= end)
].sort_values("timestamp").copy()

# Attach p95 thresholds (merge on cmdb_id, log_name)
log_window_cur = log_window_cur.merge(p95_log, on=["cmdb_id", "log_name"], how="left")

# Prepare rows limited to at most 50 rows with ISO timestamp
if log_window_cur.shape[0] > 0:
    log_window_cur_rows = log_window_cur.loc[:, ["timestamp", "log_name", "value"]].copy()
    log_window_cur_rows["timestamp"] = log_window_cur_rows["timestamp"].dt.tz_convert("UTC").dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    log_window_cur_rows = log_window_cur_rows.head(50)
else:
    log_window_cur_rows = pd.DataFrame(columns=["timestamp", "log_name", "value"])

# Summary per log_name for currencyservice in-window
def log_group_stats(df):
    p95 = df["p95_threshold"].iloc[0]
    pts = len(df)
    if pd.isna(p95):
        anom_cnt = 0
        earliest = None
    else:
        is_anom = df["value"] >= p95
        anom_cnt = int(is_anom.sum())
        earliest = None
        if anom_cnt > 0:
            earliest = pd.to_datetime(df.loc[is_anom, "timestamp"].min()).tz_convert("UTC").strftime("%Y-%m-%dT%H:%M:%SZ")
    max_val = float(df["value"].max()) if pts>0 else float("nan")
    return pd.Series({
        "p95_threshold": float(p95) if pd.notna(p95) else float("nan"),
        "points_in_window": int(pts),
        "anomaly_count": anom_cnt,
        "earliest_anomaly_ts": earliest,
        "max_value_in_window": max_val,
    })

if log_window_cur.shape[0] > 0:
    currency_log_summary = (
        log_window_cur.groupby("log_name", observed=True)
        .apply(log_group_stats)
        .reset_index()
    )
else:
    currency_log_summary = pd.DataFrame(columns=["log_name","p95_threshold","points_in_window","anomaly_count","earliest_anomaly_ts","max_value_in_window"])

# -------------------------
# Final compact outputs (variables for IPython to display)
# -------------------------
# currency_metric_kpis: one row per currencyservice kpi present in the window with requested fields
# m_window_cur.shape gives context (points total in window)
# log_window_cur_rows: up to 50 rows with timestamp ISO, log_name, value
# currency_log_summary: per log_name summary as requested

currency_metric_kpis, m_window_cur.shape, log_window_cur_rows, currency_log_summary

```
Out[5]:
```
Summary — currencyservice focused (incident window 2024-01-19T19:52:13Z → 2024-01-19T20:22:13Z)

Metric findings (per kpi_name):
- Observed KPIs in-window: cpu, diskio, latency-50, latency-90, mem, socket, workload (each had 25 points in the window).
- socket: p95=7.0, points_in_window=25, anomaly_count=12. These 12 anomalies form one consecutive anomalous run starting at 2024-01-19T20:08:00Z and lasting 12 points → likely sustained socket issue beginning ~20:08Z. max=7.0.
- cpu: p95≈48.01, points=25, anomaly_count=2, earliest anomaly 2024-01-19T20:08:00Z, max≈48.99, two short anomaly segments (earliest segment length 1).
- diskio: p95≈1.155e10, points=25, anomaly_count=2, earliest 2024-01-19T20:09:00Z, max≈1.1607e10, two short segments.
- latency-50: p95≈0.06438, points=25, anomaly_count=2, earliest 2024-01-19T20:08:00Z, max≈0.06686, two short segments.
- latency-90: p95≈0.14019, points=25, anomaly_count=2, earliest 2024-01-19T20:08:00Z, max≈0.16095, two short segments.
- mem: p95≈2.68365e8, points=25, anomaly_count=2, earliest 2024-01-19T20:14:00Z, max≈2.68427e8, two short segments.
- workload: p95≈84.39, points=25, anomaly_count=2, earliest 2024-01-19T20:05:00Z, max≈87.41, two short segments.

Logs (currencyservice):
- Rows returned for currencyservice in-window: many (sample limited to 50 rows shown). Total metric rows for currencyservice in-window = 175.
- log.error_count: global p95 = 0.0, points_in_window=25, anomaly_count=25, earliest anomaly 2024-01-19T19:55:00Z, max=0.0. (Because p95=0.0, all zero values meet >= P95 — likely a threshold artifact; treat with caution.)
- log.row_count: global p95 ≈ 2480.2, points_in_window=25, anomaly_count=2, earliest anomaly 2024-01-19T19:57:00Z, max=2541.0 — two high row_count samples above the P95.

Actionable takeaways:
- The most significant signal for currencyservice is the socket metric: a sustained run of 12 consecutive anomalous points beginning ~2024-01-19T20:08:00Z. Investigate socket errors/connection issues (network, resource exhaustion, keepalive limits).
- Secondary spikes: short spikes in cpu, diskio, latency metrics around ~20:08–20:09Z and a memory spike ~20:14Z — investigate whether these coincide with the socket problem or are downstream effects.
- Logs: treat log.error_count anomaly counts cautiously because p95=0.0 flags all zeros; instead focus on log.row_count which had two high samples (>= P95) near 19:57Z.
- Correlate with trace results (currencyservice had notable duration anomalies around ~19:59Z in earlier trace analysis) to determine causal direction (e.g., incoming request burst → socket pressure → increased latencies).

The original code execution output of IPython Kernel is also provided below for reference:

(     kpi_name  p95_threshold  points_in_window  anomaly_count   earliest_anomaly_ts  max_value_in_window  number_of_consecutive_anomaly_segments earliest_fault_start_ts  length_of_earliest_fault_in_points
0         cpu   4.801182e+01                25              2  2024-01-19T20:08:00Z         4.899151e+01                                       2    2024-01-19T20:08:00Z                                   1
1      diskio   1.155227e+10                25              2  2024-01-19T20:09:00Z         1.160703e+10                                       2    2024-01-19T20:09:00Z                                   1
2  latency-50   6.438497e-02                25              2  2024-01-19T20:08:00Z         6.686289e-02                                       2    2024-01-19T20:08:00Z                                   1
3  latency-90   1.401864e-01                25              2  2024-01-19T20:08:00Z         1.609512e-01                                       2    2024-01-19T20:08:00Z                                   1
4         mem   2.683650e+08                25              2  2024-01-19T20:14:00Z         2.684273e+08                                       2    2024-01-19T20:14:00Z                                   1
5      socket   7.000000e+00                25             12  2024-01-19T20:08:00Z         7.000000e+00                                       1    2024-01-19T20:08:00Z                                  12
6    workload   8.438783e+01                25              2  2024-01-19T20:05:00Z         8.741407e+01                                       2    2024-01-19T20:05:00Z                                   1, (175, 5),                timestamp         log_name  value
0   2024-01-19T19:55:00Z  log.error_count      0
1   2024-01-19T19:55:00Z    log.row_count   1874
2   2024-01-19T19:56:00Z  log.error_count      0
3   2024-01-19T19:56:00Z    log.row_count   2325
4   2024-01-19T19:57:00Z  log.error_count      0
..                   ...              ...    ...
45  2024-01-19T20:17:00Z    log.row_count   2268
46  2024-01-19T20:18:00Z  log.error_count      0
47  2024-01-19T20:18:00Z    log.row_count   2440
48  2024-01-19T20:19:00Z  log.error_count      0
49  2024-01-19T20:19:00Z    log.row_count    509

[50 rows x 3 columns],           log_name  p95_threshold  points_in_window  anomaly_count   earliest_anomaly_ts  max_value_in_window
0  log.error_count            0.0                25             25  2024-01-19T19:55:00Z                  0.0
1    log.row_count         2480.2                25              2  2024-01-19T19:57:00Z               2541.0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```